In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd

data = pd.read_csv('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/sample_submission.csv')
data.to_csv('submission.csv', index=False)

In [ ]:
!pip install lightgbm xgboost -q
print("Install done ✓")

In [ ]:
import os, warnings, gc
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm.notebook import tqdm

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
np.random.seed(42)
print("Imports OK ✓")

In [ ]:
# ─── PATHS (confirmed from your notebook) ───────────────────
BASE        = Path('/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup')
STEMS_DIR   = BASE / 'genres_stems'
MASHUPS_DIR = BASE / 'mashups'
ESC50_DIR   = BASE / 'ESC-50-master' / 'audio'
TEST_CSV    = BASE / 'test.csv'
OUT_DIR     = Path('/kaggle/working')
CACHE       = OUT_DIR / 'cache'
CACHE.mkdir(exist_ok=True)

# ─── AUDIO CONFIG ────────────────────────────────────────────
SR         = 22050
DURATION   = 30        # seconds per clip
N_MFCC     = 40
N_MELS     = 128
HOP        = 512
N_FFT      = 2048

# ─── TRAINING CONFIG ─────────────────────────────────────────
N_AUGMENTS           = 5    # augmented copies per original song
N_MASHUPS_PER_GENRE  = 100  # simulated cross-song mashups per genre
PSEUDO_THRESHOLD     = 0.97 # confidence cutoff for pseudo-labeling

# ─── VERIFY PATHS ────────────────────────────────────────────
print("Stems dir   :", STEMS_DIR.exists(),   STEMS_DIR)
print("Mashups dir :", MASHUPS_DIR.exists(), MASHUPS_DIR)
print("ESC-50 dir  :", ESC50_DIR.exists(),   ESC50_DIR)
print("test.csv    :", TEST_CSV.exists(),     TEST_CSV)

# List genres
genres = sorted([d.name for d in STEMS_DIR.iterdir() if d.is_dir()])
print("\nGenres found:", genres)
print("Total genres:", len(genres))

In [ ]:
# ─── FEATURE EXTRACTION ─────────────────────────────────────
# Extracts ~800+ dims per audio clip:
# MFCC+deltas, Mel, Chroma×3, Spectral, Tonnetz, ZCR, RMS, Tempo, H/P

def extract_features(y, sr=SR):
    feats = []

    # 1. MFCC + delta + delta2  (mean/std/min/max each)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP)
    d1   = librosa.feature.delta(mfcc)
    d2   = librosa.feature.delta(mfcc, order=2)
    for m in [mfcc, d1, d2]:
        feats += [m.mean(1), m.std(1), m.min(1), m.max(1)]

    # 2. Mel-spectrogram
    mel = librosa.power_to_db(
        librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP))
    feats += [mel.mean(1), mel.std(1)]

    # 3. Chroma (STFT + CQT + CENS)
    for ch in [
        librosa.feature.chroma_stft(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP),
        librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=HOP),
        librosa.feature.chroma_cens(y=y, sr=sr, hop_length=HOP),
    ]:
        feats += [ch.mean(1), ch.std(1)]

    # 4. Spectral features
    for arr in [
        librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP)[0],
        librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=HOP)[0],
        librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=HOP)[0],
        librosa.feature.spectral_flatness(y=y, hop_length=HOP)[0],
    ]:
        feats.append(np.array([arr.mean(), arr.std(), arr.min(), arr.max()]))

    sc = librosa.feature.spectral_contrast(y=y, sr=sr, hop_length=HOP)
    feats += [sc.mean(1), sc.std(1)]

    # 5. Tonnetz (tonal centroid features)
    tn = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
    feats += [tn.mean(1), tn.std(1)]

    # 6. ZCR + RMS
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=HOP)[0]
    rms = librosa.feature.rms(y=y, hop_length=HOP)[0]
    feats.append(np.array([zcr.mean(), zcr.std(), rms.mean(), rms.std()]))

    # 7. Tempo + beat strength
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP)
    onset = librosa.onset.onset_strength(y=y, sr=sr, hop_length=HOP)
    feats.append(np.array([float(tempo), onset.mean(), onset.std(), float(len(beats))]))

    # 8. Harmonic / Percussive energy ratio
    yh, yp = librosa.effects.hpss(y)
    he = float(np.mean(yh**2))
    pe = float(np.mean(yp**2))
    feats.append(np.array([he, pe, he / (pe + 1e-9)]))

    # 9. Poly features
    poly = librosa.feature.poly_features(y=y, sr=sr, order=2)
    feats += [poly.mean(1), poly.std(1)]

    return np.concatenate([f.ravel() for f in feats]).astype(np.float32)


def normalize(y):
    pk = np.abs(y).max()
    return y / pk if pk > 0 else y


print(f"Feature dim test: {extract_features(np.random.randn(SR).astype(np.float32)).shape[0]} dims ✓")

In [ ]:
# ─── NOISE CLIPS + AUGMENTATION ─────────────────────────────

def load_noise_clips(esc50_dir, sr=SR, max_clips=300):
    clips = []
    if not Path(esc50_dir).exists():
        print("ESC-50 not found — noise augmentation disabled")
        return clips
    paths = list(Path(esc50_dir).glob('*.wav'))
    np.random.shuffle(paths)
    for p in paths[:max_clips]:
        try:
            y, _ = librosa.load(p, sr=sr, mono=True)
            clips.append(y)
        except:
            pass
    print(f"Loaded {len(clips)} ESC-50 noise clips ✓")
    return clips


def mix_noise(y, noise_clips, snr_min=3, snr_max=25):
    if not noise_clips:
        return y
    noise = noise_clips[np.random.randint(len(noise_clips))]
    if len(noise) < len(y):
        noise = np.tile(noise, int(np.ceil(len(y) / len(noise))))
    start = np.random.randint(0, max(1, len(noise) - len(y)))
    ns = noise[start:start + len(y)]
    snr   = np.random.uniform(snr_min, snr_max)
    sp    = np.mean(y**2) + 1e-9
    np_   = np.mean(ns**2) + 1e-9
    scale = np.sqrt(sp / np_ / (10**(snr / 10)))
    return np.clip(y + scale * ns, -1, 1)


def augment(y, sr, noise_clips):
    out = []
    # Noise injection
    out.append(mix_noise(y, noise_clips))
    # Pitch shift ±3 semitones
    out.append(librosa.effects.pitch_shift(
        y, sr=sr, n_steps=float(np.random.uniform(-3, 3))))
    # Time stretch 0.85–1.15x
    rate = float(np.random.uniform(0.85, 1.15))
    st = librosa.effects.time_stretch(y, rate=rate)
    st = st[:len(y)] if len(st) > len(y) else np.pad(st, (0, len(y) - len(st)))
    out.append(st)
    # Volume scaling
    out.append(np.clip(y * np.random.uniform(0.5, 1.5), -1, 1))
    # Gaussian noise
    sigma = np.random.uniform(0.001, 0.015)
    out.append(np.clip(y + np.random.normal(0, sigma, len(y)), -1, 1))
    return out


noise_clips = load_noise_clips(ESC50_DIR)

In [ ]:
# ─── BUILD TRAINING FEATURES ────────────────────────────────
# Extracts features from all 1000 songs (100/genre × 10 genres)
# + N_AUGMENTS augmented versions each
# Cached to /kaggle/working/cache/ so re-runs are fast

def build_train_features():
    X, y_lab = [], []
    for genre in genres:
        song_dirs = sorted([d for d in (STEMS_DIR / genre).iterdir() if d.is_dir()])
        print(f"  {genre}: {len(song_dirs)} songs", flush=True)
        for sd in tqdm(song_dirs, desc=f"  {genre}", leave=False):
            try:
                # Load all 4 stems
                stems = {}
                for sn in ['drums', 'vocals', 'bass', 'others']:
                    p = sd / f'{sn}.wav'
                    if p.exists():
                        yy, _ = librosa.load(p, sr=SR, duration=DURATION, mono=True)
                        stems[sn] = yy

                if not stems:
                    continue

                # Mix stems
                L = max(len(v) for v in stems.values())
                mixed = normalize(sum(np.pad(v, (0, L - len(v))) for v in stems.values()))

                # Original features: mixed + each stem
                feat = np.concatenate([extract_features(mixed)] +
                                      [extract_features(v) for v in stems.values()])
                X.append(feat); y_lab.append(genre)

                # Augmented versions (mixed only — faster)
                for _ in range(N_AUGMENTS):
                    for aug_y in augment(mixed, SR, noise_clips):
                        X.append(extract_features(normalize(aug_y)))
                        y_lab.append(genre)

            except Exception as e:
                pass  # skip bad files

    return np.array(X, dtype=np.float32), np.array(y_lab)


if (CACHE / 'X_train.npy').exists():
    print("Loading cached training features...")
    X_train = np.load(CACHE / 'X_train.npy')
    y_train = np.load(CACHE / 'y_train.npy', allow_pickle=True)
    print(f"  Loaded: {X_train.shape}")
else:
    print("Extracting training features (~60–90 min on Kaggle)...")
    X_train, y_train = build_train_features()
    np.save(CACHE / 'X_train.npy', X_train)
    np.save(CACHE / 'y_train.npy', y_train)
    print(f"Done! {X_train.shape}")

In [ ]:
# ─── SIMULATED MASHUP FEATURES ──────────────────────────────
# KEY INSIGHT: Test data = stems from DIFFERENT songs (same genre) + ESC-50 noise
# We replicate this exact process to bridge the train→test distribution gap

def build_mashup_sim():
    X, y_lab = [], []
    stem_names = ['drums', 'vocals', 'bass', 'others']

    for genre in genres:
        song_dirs = sorted([d for d in (STEMS_DIR / genre).iterdir() if d.is_dir()])

        # Preload all stems for this genre
        all_stems = {s: [] for s in stem_names}
        for sd in song_dirs:
            for sn in stem_names:
                p = sd / f'{sn}.wav'
                if p.exists():
                    try:
                        yy, _ = librosa.load(p, sr=SR, duration=DURATION, mono=True)
                        all_stems[sn].append(yy)
                    except:
                        pass

        print(f"  {genre}: simulating {N_MASHUPS_PER_GENRE} mashups...", flush=True)
        for _ in tqdm(range(N_MASHUPS_PER_GENRE), desc=f"  {genre}", leave=False):
            try:
                # Pick ONE stem from each instrument from DIFFERENT random songs
                parts = []
                for sn in stem_names:
                    pool = all_stems[sn]
                    if pool:
                        parts.append(pool[np.random.randint(len(pool))])

                if not parts:
                    continue

                L = max(len(p) for p in parts)
                mixed = normalize(sum(np.pad(p, (0, L - len(p))) for p in parts))

                # Add ESC-50 noise (same as test construction)
                mixed = normalize(mix_noise(mixed, noise_clips, snr_min=3, snr_max=25))

                X.append(extract_features(mixed))
                y_lab.append(genre)
            except:
                pass

    return np.array(X, dtype=np.float32), np.array(y_lab)


if (CACHE / 'X_mashup.npy').exists():
    print("Loading cached mashup-sim features...")
    X_mashup = np.load(CACHE / 'X_mashup.npy')
    y_mashup = np.load(CACHE / 'y_mashup.npy', allow_pickle=True)
    print(f"  Loaded: {X_mashup.shape}")
else:
    print("Building simulated mashup features (~30 min)...")
    X_mashup, y_mashup = build_mashup_sim()
    np.save(CACHE / 'X_mashup.npy', X_mashup)
    np.save(CACHE / 'y_mashup.npy', y_mashup)
    print(f"Done! {X_mashup.shape}")

# Combine everything
X_all = np.vstack([X_train, X_mashup])
y_all = np.concatenate([y_train, y_mashup])
print(f"\nTotal training: {X_all.shape[0]} samples, {X_all.shape[1]} features")
gc.collect()

In [ ]:
# ─── EXTRACT TEST FEATURES ──────────────────────────────────

def extract_test_features():
    df = pd.read_csv(TEST_CSV)
    print(f"Test samples: {len(df)}")
    print(df.head())

    # Find the filename column (not 'id')
    fname_col = [c for c in df.columns if c.lower() != 'id'][0]
    print(f"Filename column: '{fname_col}'")

    X_test, ids = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Test"):
        # Try filename as-is, then with .wav extension
        for suffix in [str(row[fname_col]), f"{row['id']}.wav"]:
            audio_path = MASHUPS_DIR / suffix
            if audio_path.exists():
                break

        try:
            y, _ = librosa.load(audio_path, sr=SR, duration=DURATION, mono=True)
            y = normalize(y)
            X_test.append(extract_features(y))
            ids.append(row['id'])
        except Exception as e:
            print(f"  FAILED id={row['id']}: {e}")

    return np.array(X_test, dtype=np.float32), np.array(ids), df


if (CACHE / 'X_test.npy').exists():
    print("Loading cached test features...")
    X_test   = np.load(CACHE / 'X_test.npy')
    test_ids = np.load(CACHE / 'test_ids.npy', allow_pickle=True)
    df_test  = pd.read_csv(TEST_CSV)
    print(f"  Loaded: {X_test.shape}")
else:
    print("Extracting test features (~20 min)...")
    X_test, test_ids, df_test = extract_test_features()
    np.save(CACHE / 'X_test.npy',   X_test)
    np.save(CACHE / 'test_ids.npy', test_ids)
    print(f"Done! {X_test.shape}")

In [ ]:
# ─── SCALE + CROSS-VALIDATION CHECK ─────────────────────────

le     = LabelEncoder()
y_enc  = le.fit_transform(y_all)
n_cls  = len(le.classes_)
scaler = RobustScaler()
X_sc   = scaler.fit_transform(X_all)
X_ts   = scaler.transform(X_test)

print(f"Classes ({n_cls}): {le.classes_}")
print(f"Feature dims: {X_sc.shape[1]}")
print(f"Training samples: {X_sc.shape[0]}")

# Quick 5-fold CV with LightGBM to sanity-check
print("\nRunning 5-fold CV sanity check...")
lgb_quick = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=63,
    random_state=42, verbose=-1
)
cv = cross_val_score(
    lgb_quick, X_sc, y_enc,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='f1_macro', n_jobs=-1
)
print(f"LGB CV Macro F1: {cv.mean():.4f} ± {cv.std():.4f}")
print("(CV on mixed train — actual test score will differ due to mashup gap)")

In [ ]:
# ─── TRAIN ENSEMBLE (5 models) ──────────────────────────────

print("Training LightGBM (best single model)...")
clf_lgb = lgb.LGBMClassifier(
    n_estimators=1000, learning_rate=0.03, num_leaves=127,
    max_depth=8, min_child_samples=10,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    class_weight='balanced', random_state=42, verbose=-1
)
clf_lgb.fit(X_sc, y_enc)
print("  LGB done ✓")

print("Training XGBoost...")
clf_xgb = xgb.XGBClassifier(
    n_estimators=800, learning_rate=0.03, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    gamma=0.1, min_child_weight=3,
    eval_metric='mlogloss', tree_method='hist',
    random_state=42, use_label_encoder=False
)
clf_xgb.fit(X_sc, y_enc)
print("  XGB done ✓")

print("Training ExtraTrees...")
clf_et = ExtraTreesClassifier(
    n_estimators=500, class_weight='balanced',
    random_state=42, n_jobs=-1
)
clf_et.fit(X_sc, y_enc)
print("  ET done ✓")

print("Training SVM (calibrated, may take 5–10 min)...")
clf_svm = CalibratedClassifierCV(
    SVC(kernel='rbf', C=10, gamma='scale',
        class_weight='balanced', random_state=42),
    cv=3
)
clf_svm.fit(X_sc, y_enc)
print("  SVM done ✓")

print("Training MLP...")
clf_mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128), activation='relu',
    alpha=1e-3, learning_rate='adaptive',
    max_iter=300, random_state=42
)
clf_mlp.fit(X_sc, y_enc)
print("  MLP done ✓")

models = {
    'lgb': clf_lgb,
    'xgb': clf_xgb,
    'et' : clf_et,
    'svm': clf_svm,
    'mlp': clf_mlp,
}

# Weights: LGB/XGB best for tabular audio features
WEIGHTS = {'lgb': 3.0, 'xgb': 2.5, 'et': 1.5, 'svm': 1.0, 'mlp': 1.0}
print("\nAll 5 models trained ✓")

In [ ]:
# ─── SOFT-VOTING PREDICT HELPER ─────────────────────────────

def ensemble_proba(X_scaled, weights=WEIGHTS):
    total_w = sum(weights.values())
    proba = np.zeros((len(X_scaled), n_cls), dtype=np.float64)
    for name, clf in models.items():
        w = weights.get(name, 1.0) / total_w
        proba += w * clf.predict_proba(X_scaled)
    return proba

def ensemble_predict(X_scaled):
    proba = ensemble_proba(X_scaled)
    return le.inverse_transform(np.argmax(proba, 1)), proba

# Initial predictions
y_pred_init, proba_init = ensemble_predict(X_ts)
print(f"Initial prediction distribution:")
print(pd.Series(y_pred_init).value_counts().to_string())
conf = proba_init.max(1)
print(f"\nMean confidence: {conf.mean():.3f}")
print(f">0.90 confidence: {(conf>0.90).sum()}/{len(conf)} ({100*(conf>0.90).mean():.1f}%)")

In [ ]:
# ─── PSEUDO-LABELING (semi-supervised) ──────────────────────
# High-confidence test predictions → added to training → retrain
# This directly exploits the unlabeled test set

X_tr_cur = X_all.copy()
y_tr_cur = y_all.copy()

for iteration in range(3):
    print(f"\n══ Pseudo-label iteration {iteration+1}/3 ══")

    # Scale + encode current training set
    sc_cur = RobustScaler().fit(X_tr_cur)
    le_cur = LabelEncoder().fit(y_tr_cur)
    X_sc_cur = sc_cur.transform(X_tr_cur)
    y_enc_cur = le_cur.transform(y_tr_cur)
    X_ts_cur  = sc_cur.transform(X_test)

    # Refit LGB + XGB + ET (fast models)
    clf_lgb.set_params(n_estimators=800); clf_lgb.fit(X_sc_cur, y_enc_cur)
    clf_xgb.set_params(n_estimators=600); clf_xgb.fit(X_sc_cur, y_enc_cur)
    clf_et.fit(X_sc_cur, y_enc_cur)

    # Update scaler/encoder reference
    scaler = sc_cur; le = le_cur; X_ts = X_ts_cur; n_cls = len(le.classes_)

    # Predict on test
    proba = ensemble_proba(X_ts_cur)
    max_p = proba.max(1)
    mask  = max_p >= PSEUDO_THRESHOLD

    print(f"  Confident samples (≥{PSEUDO_THRESHOLD}): {mask.sum()}/{len(X_test)}")
    if mask.sum() == 0:
        print("  No confident samples — stopping early")
        break

    pseudo_labels = le.inverse_transform(np.argmax(proba[mask], 1))
    print(f"  Pseudo label distribution: {pd.Series(pseudo_labels).value_counts().to_dict()}")

    X_tr_cur = np.vstack([X_tr_cur, X_test[mask]])
    y_tr_cur = np.concatenate([y_tr_cur, pseudo_labels])
    print(f"  Training set: {len(y_tr_cur)} samples (+{mask.sum()})")

print("\nPseudo-labeling complete ✓")

In [ ]:
# ─── FINAL RETRAIN + SUBMISSION ─────────────────────────────

print("Final retraining on full pseudo-labeled dataset...")
sc_final  = RobustScaler().fit(X_tr_cur)
le_final  = LabelEncoder().fit(y_tr_cur)
X_sc_fin  = sc_final.transform(X_tr_cur)
y_enc_fin = le_final.transform(y_tr_cur)
X_ts_fin  = sc_final.transform(X_test)
n_cls_fin = len(le_final.classes_)

# Full retrain all 5 models
clf_lgb.set_params(n_estimators=1000)
clf_lgb.fit(X_sc_fin, y_enc_fin); print("  LGB ✓")
clf_xgb.set_params(n_estimators=800)
clf_xgb.fit(X_sc_fin, y_enc_fin); print("  XGB ✓")
clf_et.fit(X_sc_fin, y_enc_fin);  print("  ET  ✓")
clf_svm.fit(X_sc_fin, y_enc_fin); print("  SVM ✓")
clf_mlp.fit(X_sc_fin, y_enc_fin); print("  MLP ✓")

# Final predictions
n_cls = n_cls_fin
def ensemble_proba_fin(X_scaled, weights=WEIGHTS):
    total_w = sum(weights.values())
    proba = np.zeros((len(X_scaled), n_cls), dtype=np.float64)
    for name, clf in models.items():
        w = weights.get(name, 1.0) / total_w
        proba += w * clf.predict_proba(X_scaled)
    return proba

proba_fin = ensemble_proba_fin(X_ts_fin)
y_pred_fin = le_final.inverse_transform(np.argmax(proba_fin, 1))

# ─── BUILD SUBMISSION CSV ────────────────────────────────────
sub = pd.DataFrame({'id': test_ids, 'genre': y_pred_fin})
sub = df_test[['id']].merge(sub, on='id', how='left')
fallback = pd.Series(y_pred_fin).value_counts().index[0]
sub['genre'] = sub['genre'].fillna(fallback)

out_path = OUT_DIR / 'submission.csv'
sub.to_csv(out_path, index=False)

# ─── SUMMARY ─────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"✅ Submission saved: {out_path}")
print(f"   Rows: {len(sub)}")
print(f"\nGenre distribution:")
print(sub['genre'].value_counts().to_string())

conf = proba_fin.max(1)
print(f"\nConfidence stats:")
print(f"  Mean : {conf.mean():.4f}")
print(f"  Min  : {conf.min():.4f}")
print(f"  >0.95: {(conf>0.95).sum()}/{len(conf)} ({100*(conf>0.95).mean():.1f}%)")
print(f"  >0.99: {(conf>0.99).sum()}/{len(conf)} ({100*(conf>0.99).mean():.1f}%)")

print(f"\nSample rows:")
print(sub.head(10).to_string(index=False))
print(f"{'='*50}")

In [ ]:
# ─── VERIFY SUBMISSION FORMAT ───────────────────────────────
sub_check = pd.read_csv(OUT_DIR / 'submission.csv')
valid_genres = {'blues','classical','country','disco','hiphop',
                'jazz','metal','pop','reggae','rock'}

print("Submission shape:", sub_check.shape)
print("Columns:", sub_check.columns.tolist())
print("Null values:", sub_check.isnull().sum().to_dict())
print("Invalid genres:", set(sub_check['genre'].unique()) - valid_genres)
print("\n✅ Submission is valid!" if len(set(sub_check['genre'].unique()) - valid_genres) == 0
      else "⚠️ Invalid genre labels found!")
print("\nHead:")
print(sub_check.head())